##### MHKiT Hindcast Resource Assessment

Original Author: Hannah Mankle 

This Jupyter notebook uses NREL's MHKiT toolbox to perform resource assessments using data from the MHKiT Hindcast Database. https://registry.opendata.aws/wpto-pds-us-wave/

All files are virtual buoys along coastlines of the US with 3-hour temporal resolutions to a 200m spatial resoution. PWS data is pulled with the 1-hour temporal resoulions which includes direction wave spectrum data in HDF5 format.

Currently MHKiT pulls the Hindcast data from AWS and cannot complete a request for 10+ years of data. Data for the locations beflow have been pulled from AWS and saved to .CSV files to complete assessments that meet IEC standards. Figures on these locations can be found in the Graphics folder under each location. General information from this assessment can be found in the Resource Conditions folder in the parent directory.

Note: AK, HI & Miami do not have mean zero-crossing period in dataset -> do not calculate wave steepness. AK does not have files for 1979,1991,2010

The current version of MHKiT used to run this code is a fork of the main MHKiT repository. Please install the version of MHKiT from https://github.com/camirmas/MHKiT-Python if you want to pull directly from the local .h5 files. 

In [1]:
import os
config_path = os.path.expanduser("~/.hscfg")

print("Config path:", config_path)
print("File exists:", os.path.isfile(config_path))

Config path: C:\Users\gupta/.hscfg
File exists: True


In [2]:
import h5pyd

with h5pyd.File(
    "/nrel/US_wave/v1.0.1/Gulf_of_Mexico_and_Puerto_Rico/GOM_PR_2010.h5",
    "r",
    timeout=120
) as f:
    print(list(f.keys()))

['coordinates', 'depth', 'direction_of_maximum_directionally_resolved_wave_power', 'directionality_coefficient', 'energy_period', 'mean_absolute_period', 'mean_wave_direction', 'mean_zero-crossing_period', 'meta', 'missing_time_index', 'omnidirectional_wave_power', 'peak_period', 'peak_wave_direction', 'significant_wave_height', 'spectral_width', 'time_index']


In [3]:
import requests

url = "https://developer.nlr.gov/api/hsds/"

try:
    response = requests.get(url, timeout=10)
    print("HTTP status:", response.status_code)
except requests.exceptions.Timeout:
    print("Server did not respond within 10 seconds.")
except requests.exceptions.RequestException as e:
    print(type(e).__name__, str(e))

HTTP status: 403


In [4]:
# Importing necessary python packages

from mhkit import wave
from mhkit import power, utils

import matplotlib.pyplot as plt
from scipy import stats
import pandas as pd
import numpy as np
import folium
import requests
import os
import h5pyd
from pathlib import Path
# from windrose import WindroseAxes, WindAxes
import matplotlib.cm as cm
import matplotlib.dates as dates
from datetime import time, datetime  
import xarray as xr

Load in CSV from data_pull

In [5]:
locations = pd.read_csv("nearest_locations.csv")

names = []
lats = []
lons = []
file_paths = []
main_folders = []
secondary_folders = []
graphics_paths=[]
nearest_lat = []
nearest_lon = []
index = []
regions = []

for _, location in locations.iterrows():
    
    names.append(location["location_name"])
    lats.append(location["latitude"])
    lons.append(location["longitude"])
    file_paths.append(location["file_path"])
    main_folders.append(location['main_folder'])
    secondary_folders.append(location['secondary_folder'])
    nearest_lat.append(location["Nearest_Latitude"])
    nearest_lon.append(location["Nearest_Longitude"])
    index.append(location["Grid_Index"])
    regions.append(location["region"])

all_lat_lon = np.column_stack((lats,lons))
print(all_lat_lon)
# Check that it loaded correctly

# print(names)
# print(lats)
# print(lons)
# print(file_paths)
# print(nearest_lat)
# print(nearest_lon)
# print(index)
# print(regions)
# print(main_folders)
# print(secondary_folders)

[[  29.071107  -90.227879]
 [  28.912614  -95.269769]
 [  29.266923  -94.571306]
 [  46.225134 -123.97398 ]
 [  46.248895 -124.011479]
 [  46.185001 -124.183709]
 [  36.825459  -75.781913]
 [  36.872679  -75.868236]
 [  36.885472  -75.888675]]


Combine latitudes and longitudes as pairs

save data for each parameter

In [6]:
from mhkit import wave
lat_lon_pairs = np.column_stack((lats,lons)).tolist()

parameters = [
    "significant_wave_height",
    "peak_period",
    "energy_period",
    "omni-directional_wave_power",
    "mean_wave_direction",
    "spectral_width",
    "directionality_coefficient"
]

wave.io.hindcast.hindcast.request_wpto_point_data is not working for any locations

**Establish Path to Data in a Loop**

Choose the number of years to cycle through as well

Creates a location key to allow easy run through of each locations data

In [33]:
data_paths = []
years = [1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010] #enter years as comma seperated list
years_per_loc = len(years)

for _, location in locations.iterrows():

    file_path = str(location["file_path"])
    main_folder = str(location["main_folder"])
    secondary_folder = str(location["secondary_folder"])

    for year in years:
        data_path = file_path + main_folder + secondary_folder + str(year) + ".h5"
        data_paths.append(data_path)

print(data_paths)

location_key = np.repeat(np.arange(len(locations)), len(years))
print(location_key)

['/nrel/US_wave/v1.0.1/Gulf_of_Mexico_and_Puerto_Rico/GOM_PR_1999.h5', '/nrel/US_wave/v1.0.1/Gulf_of_Mexico_and_Puerto_Rico/GOM_PR_2000.h5', '/nrel/US_wave/v1.0.1/Gulf_of_Mexico_and_Puerto_Rico/GOM_PR_2001.h5', '/nrel/US_wave/v1.0.1/Gulf_of_Mexico_and_Puerto_Rico/GOM_PR_2002.h5', '/nrel/US_wave/v1.0.1/Gulf_of_Mexico_and_Puerto_Rico/GOM_PR_2003.h5', '/nrel/US_wave/v1.0.1/Gulf_of_Mexico_and_Puerto_Rico/GOM_PR_2004.h5', '/nrel/US_wave/v1.0.1/Gulf_of_Mexico_and_Puerto_Rico/GOM_PR_2005.h5', '/nrel/US_wave/v1.0.1/Gulf_of_Mexico_and_Puerto_Rico/GOM_PR_2006.h5', '/nrel/US_wave/v1.0.1/Gulf_of_Mexico_and_Puerto_Rico/GOM_PR_2007.h5', '/nrel/US_wave/v1.0.1/Gulf_of_Mexico_and_Puerto_Rico/GOM_PR_2008.h5', '/nrel/US_wave/v1.0.1/Gulf_of_Mexico_and_Puerto_Rico/GOM_PR_2009.h5', '/nrel/US_wave/v1.0.1/Gulf_of_Mexico_and_Puerto_Rico/GOM_PR_2010.h5', '/nrel/US_wave/v1.0.1/Gulf_of_Mexico_and_Puerto_Rico/GOM_PR_1999.h5', '/nrel/US_wave/v1.0.1/Gulf_of_Mexico_and_Puerto_Rico/GOM_PR_2000.h5', '/nrel/US_wave/v1.0

In [34]:
all_data = []

for i, path in enumerate(data_paths):

    loc_idx = location_key[i]
    grid_index = int(index[loc_idx])

    location_data = {}

    # with h5pyd.File(data_paths[0], "r") as f:
    #     print(list(f.keys()))

    with h5pyd.File(path, "r") as f:

        for parameter in parameters:

            dataset_name = parameter

            if parameter == "omni-directional_wave_power":
                if "omnidirectional_wave_power" in f:
                    dataset_name = "omnidirectional_wave_power"

            location_data[parameter] = f[dataset_name][:, grid_index]

        location_data["time_index"] = f["time_index"][:]

    all_data.append(location_data)

**Create Path to Save Data**

**Import Data from MHKit CSV**

Data is measured every 3 hours 

In [35]:
all_clean_data = []

for i, lat_lon in enumerate(all_lat_lon):

    yearly_data = []

    for j, year in enumerate(years):

        # Find the correct location-year data
        data_idx = i * years_per_loc + j

        # Convert HDF5 dictionary into DataFrame
        data = pd.DataFrame(all_data[data_idx])

        # Create datetime index
        data["Datetime"] = pd.to_datetime(data["time_index"].str.decode("utf-8"))
        data = data.set_index("Datetime")
        data = data.drop(columns=["time_index"])

        # Clean column names
        data.columns = (
            data.columns.str.strip()
            .str.lower()
            .str.replace(" ", "_")
        )

        # Remove entirely empty columns
        data = data.dropna(axis=1, how="all")

        # Filter wave heights
        data = data[data["significant_wave_height"] <= 20]

        # Remove missing values
        data = data.dropna()

        # Sort by date
        data.sort_index(inplace=True)

        # Calculate wave power only if not already available
        if "omni-directional_wave_power" not in data.columns:
            data["omni-directional_wave_power"] = (
                490.6
                * data["significant_wave_height"]**2
                * data["energy_period"]
            )

        yearly_data.append(data)

    # Combine all years for this location
    location_data = pd.concat(yearly_data)
    location_data.sort_index(inplace=True)

    all_clean_data.append(location_data)

**Make folders for the data**

In [36]:
all_csv_paths = []
graphics_folders = []
for i, name in enumerate(names):

    name_csv = name + "_CSV"

    path = os.getcwd()
    parent = os.path.abspath(os.path.join(path, os.pardir))

    loc_folder = os.path.join(parent, "location", name_csv)
    wave_folder = os.path.join(loc_folder, "Wave")
    graphics_folder = os.path.join(loc_folder, "Graphics")

    os.makedirs(wave_folder, exist_ok=True)
    os.makedirs(graphics_folder, exist_ok=True)

    filename_total = os.path.join(
        wave_folder, name + "_virtual_buoy_All.csv"
    )

    all_clean_data[i].to_csv(filename_total)

    print(f"Saved: {filename_total}")

    print(f"done")

    all_csv_paths.append(filename_total)
    graphics_folders.append(graphics_folder)



Saved: c:\Users\gupta\projects\mhkit-repo\location\Belle_Pass_CSV\Wave\Belle_Pass_virtual_buoy_All.csv
done
Saved: c:\Users\gupta\projects\mhkit-repo\location\Freeport_Entrance_CSV\Wave\Freeport_Entrance_virtual_buoy_All.csv
done
Saved: c:\Users\gupta\projects\mhkit-repo\location\Galveston_Bay_CSV\Wave\Galveston_Bay_virtual_buoy_All.csv
done
Saved: c:\Users\gupta\projects\mhkit-repo\location\Desdemona_Sands_CSV\Wave\Desdemona_Sands_virtual_buoy_All.csv
done
Saved: c:\Users\gupta\projects\mhkit-repo\location\Clatsop_Spit_CSV\Wave\Clatsop_Spit_virtual_buoy_All.csv
done
Saved: c:\Users\gupta\projects\mhkit-repo\location\Columbia_River_CSV\Wave\Columbia_River_virtual_buoy_All.csv
done
Saved: c:\Users\gupta\projects\mhkit-repo\location\Chesapeake_Bay_Mouth_CSV\Wave\Chesapeake_Bay_Mouth_virtual_buoy_All.csv
done
Saved: c:\Users\gupta\projects\mhkit-repo\location\Chesapeake_Bay_Southern_CSV\Wave\Chesapeake_Bay_Southern_virtual_buoy_All.csv
done
Saved: c:\Users\gupta\projects\mhkit-repo\locati

In [37]:
# J_kW = data['omni-directional_wave_power'] / 1000   # W/m → kW/m
monthly_J_all = []
annual_J_all = []
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
# by_month = J_kW.groupby(J_kW.index.month)

for i, name in enumerate(names):

    data = all_clean_data[i]

    J_kW = data['omni-directional_wave_power'] / 1000
    by_month = J_kW.groupby(J_kW.index.month)


    monthly_J = pd.DataFrame({
        'Mean (kW/m)':   by_month.mean(),
        'Median (kW/m)': by_month.median(),
        '10th pct':      by_month.quantile(0.1),
        '90th pct':      by_month.quantile(0.9),
    })
    monthly_J.index = month_names
    monthly_J = monthly_J.round(3)

    annual_mean_J = J_kW.mean()

    monthly_J_all.append(monthly_J)
    annual_J_all.append(annual_mean_J)

    print(f"\nLocation: {name}")
    print(monthly_J)
    print(f"\nOverall mean J: {J_kW.mean():.3f} kW/m")


Location: Belle_Pass
     Mean (kW/m)  Median (kW/m)  10th pct  90th pct
Jan        1.964          0.980     0.145     5.015
Feb        1.868          1.001     0.147     4.907
Mar        2.387          1.343     0.212     6.386
Apr        2.274          1.492     0.266     5.507
May        1.841          1.170     0.230     4.418
Jun        1.207          0.653     0.157     2.739
Jul        0.916          0.434     0.148     1.539
Aug        0.622          0.283     0.070     1.163
Sep        1.250          0.406     0.085     2.680
Oct        1.403          0.530     0.077     3.887
Nov        1.602          0.663     0.094     4.769
Dec        2.011          0.958     0.142     5.695

Overall mean J: 1.610 kW/m

Location: Freeport_Entrance
     Mean (kW/m)  Median (kW/m)  10th pct  90th pct
Jan        3.356          2.284     0.536     7.293
Feb        3.144          2.196     0.623     6.929
Mar        3.632          2.555     0.668     7.391
Apr        4.029          2.963     0

**Setup for average annual energy matrix and monthly averages**

In [38]:
# Setup
Hm0s = []
Tes = []
Js = []
Tps = []
ds = []
Ds = []

for data in all_clean_data:

    Hm0 = data['significant_wave_height']
    Te = data['energy_period']
    J = data['omni-directional_wave_power']
    Tp = data['peak_period']
    d = data['directionality_coefficient']
    data['J_tm'] = d*J
    J_tm = data['J_tm'].round(2)

    #Tz = data['mean_zero-crossing_period']
    D = pd.concat([Hm0, Tp, Te, J, J_tm], axis=1)

    Hm0s.append(Hm0)
    Tes.append(Te)
    Js.append(J)
    Tps.append(Tp)
    ds.append(d)
    Ds.append(D)
    print(f"Done\n")



Done

Done

Done

Done

Done

Done

Done

Done

Done



In [39]:
# # Only run for West Coast locations

# # Calculate wave steepness
# D['Sm'] = Hm0 / (9.81/(2*np.pi) * Tz**2)

# # Drop any NaNs created from the calculation of Hm0 or Te
# D.dropna(inplace=True)
# # Sort the DateTime index
# D.sort_index(inplace=True)
# # D.rename(index={0: "Hm0", 1: "Te", 2: "Tp", 3: "J", 4: "Tz"})

**Average Annaul Energy Matrix w/ Resource Frequency**

Plot Average Energy Flux Matrix (Looped)

In [40]:
# Plot Average Energy Flux Matrix (Looped)
# Setting the bins for the resource frequency and power distribution
Hm0_bin_size = 0.5
Hm0_edges = np.arange(0,7+Hm0_bin_size,Hm0_bin_size)
Te_bin_size = 1
Te_edges = np.arange(0, 15+Te_bin_size,Te_bin_size)
#fig = plt.figure(figsize=(12, 8))

for i, D in enumerate(Ds):
    graphics_folder = graphics_folders[i]
    fig = wave.graphics.plot_avg_annual_energy_matrix(D['significant_wave_height'], D['energy_period'], 
                        D['omni-directional_wave_power'], time_index=None, Hm0_edges=Hm0_edges, Te_edges=Te_edges)

    plt.title(f"{names[i]} - Average Annual Energy Flux Matrix")

    # Needs to be Updated
    figure = plt.gcf()
    plt.set_cmap('viridis')
    #plt.xticks([1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20])
    #plt.yticks([0.25,0.5,0.75,1.0,1.25,1.5,1.75,2.0,2.25,2.5,2.75,3.0,3.25,3.5,3.75,4.0])
    #plt.grid()
    # exporting plot to png
    figure.set_size_inches(8, 6)

    fig_path = os.path.join(graphics_folder,'AvgEnergyMatrix.png') 
    plt.savefig(fig_path,format="png",dpi=100,bbox_inches='tight')
    plt.close(figure)

Create Average Annual Energy Flux Matrix Using Mean, convert to kW (Looped)

In [41]:
for i, D in enumerate(Ds):
    # Create average annual energy flux matrix using mean, convert to kW
    JM_mean = wave.performance.wave_energy_flux_matrix(D['significant_wave_height'], D['energy_period'], 
                        D['omni-directional_wave_power']/1000, "mean", Hm0_bins=Hm0_edges, Te_bins=Te_edges)
    #print(JM_mean.shape)

    # convert to x_array
    JM_xr = xr.DataArray(JM_mean, dims=("Hm0", "Te"), coords={"Hm0":Hm0_edges, "Te":Te_edges})
    JM_xr.plot() 
    plt.title(f"{names[i]} - Mean Wave Energy Flux Matrix")
    fig_path = os.path.join(graphics_folders[i], 'MeanWaveEnergyMatrix.png')
    plt.savefig(fig_path,format="png",dpi=100,bbox_inches='tight')
    plt.close()


**JDP Matrix, Check**

JPD Matrix, Check (Looped)

In [42]:
for i, D in enumerate(Ds):
    JPD = wave.performance.capture_length_matrix(
        D['significant_wave_height'], D['energy_period'], np.zeros(len(D['significant_wave_height'])), "frequency",Hm0_bins=Hm0_edges, Te_bins=Te_edges)
    # JPD.plot()   

    # Calculate the sum of the joint probability distribution, this should be 1
    print("JPD Sum: ", JPD.sum().sum())

    bin_count = wave.performance.capture_length_matrix(
        D['significant_wave_height'], D['energy_period'], np.zeros(len(D['significant_wave_height'])), "count",Hm0_bins=Hm0_edges, Te_bins=Te_edges)

    # print("Bin Count Matrix:\n", bin_count)
    print("Bin Count Sum: ", bin_count.sum().sum())

    # convert to x_array
    JDP_xr = xr.DataArray(JPD, dims=("Hm0", "Te"), coords={"Hm0":Hm0_edges, "Te":Te_edges})
    JDP_xr.plot() 

    plt.title(f"{names[i]} - Joint Probability Distribution")
    fig_path = os.path.join(graphics_folders[i], 'JointProbabilityDistribution.png')
    plt.savefig(fig_path,format="png",dpi=100,bbox_inches='tight')
    plt.close()


JPD Sum:  1.0
Bin Count Sum:  35064.0
JPD Sum:  1.0
Bin Count Sum:  35064.0
JPD Sum:  1.0
Bin Count Sum:  35064.0
JPD Sum:  1.0
Bin Count Sum:  35064.0
JPD Sum:  1.0
Bin Count Sum:  35064.0
JPD Sum:  1.0
Bin Count Sum:  35064.0
JPD Sum:  1.0
Bin Count Sum:  35041.0
JPD Sum:  1.0
Bin Count Sum:  35041.0
JPD Sum:  1.0
Bin Count Sum:  35041.0


**Monthly Averages (Looped)**

In [43]:
for j, D in enumerate(Ds):
    # Plot monthly averages

    QoIs = D.keys()
    fig, axs = plt.subplots(len(QoIs)-1,1, figsize=(8, 12), sharex=True)

    months=D.index.month
    data_group=D.groupby(months)

    stdd = data_group.std()

    results = []
    #shade between 25% and 75%
    for i in range(len(QoIs)-1):
        QoI = QoIs[i]

        axs[i].plot(data_group.median()[QoI], marker='.',label='50th percentile')
        axs[i].plot(data_group.mean()[QoI], marker='*',label='mean')
        
        axs[i].plot(data_group.mean()[QoI]+stdd[QoI],marker='+',label='mean+std')
        axs[i].plot(data_group.mean()[QoI]-stdd[QoI],marker='o',label='mean-std')
        

        axs[i].fill_between(months.unique(),
                            data_group.quantile(q=.1)[QoI],
                            data_group.quantile(q=.9)[QoI],
                            alpha=0.2,
                            label='10-90th percentile')
        axs[i].grid()

        mx = data_group.median()[QoI].max()
        mx_month= data_group.median()[QoI].argmax()+1
        mx_avg = data_group.mean()[QoI].max()
        mx_avg_month = data_group.mean()[QoI].argmax()+1
        mn = data_group.median()[QoI].min()
        mn_month= data_group.median()[QoI].argmin()+1
        mn_avg =  data_group.mean()[QoI].min()
        mn_avg_month= data_group.mean()[QoI].argmin()+1
        results.append({
            'Parameter': QoI,
            'Max Median': round(mx, 4),
            'Month (Max Median)': mx_month,
            'Min Median': round(mn, 4),
            'Month (Min Median)': mn_month,
            'Max Mean': round(mx_avg, 4),
            'Month (Max Mean)': mx_avg_month,
            'Min Mean': round(mn_avg, 4),
            'Month (Min Mean)': mn_avg_month
        })
        print('--------------------------------------------')
        print(f'{QoI} max:{np.round(mx,4)}, month: {mx_month}')
        print(f'{QoI} max mean:{np.round(mx_avg,4)}, month: {mx_avg_month}')
        print(f'{QoI} min:{np.round(mn,4)}, month: {mn_month}')
        print(f'{QoI} min mean{np.round(mn_avg,4)}, month: {mn_avg_month}')
        plt.setp(axs[i], ylabel = QoIs[i])
  

    plt.setp(axs[-1], xlabel='Month')
    plt.setp(axs[0], ylabel=f' $Hsig$ [m]')
    plt.setp(axs[1], ylabel=f'$Te$  [s]')
    plt.setp(axs[2], ylabel=f'$J$  [W/m]')
    #plt.setp(axs[3], ylabel=r'$J_{\Theta jmax}$ [W/m]')

    #axs[3].ticklabel_format(axis='y',style='sci',scilimits=(1,4))
    axs[1].legend(loc='center left', bbox_to_anchor=(1, 0.5))
    axs[2].ticklabel_format(axis='y',style='sci',scilimits=(1,5))


    plt.tight_layout()

    summary_table = pd.DataFrame(results)
    parameter_names = {'significant_wave_height': 'Hm0 (m)',
                       'peak_period': 'Tp (s)', 'energy_period': 'Te (s)',
                       'omni-directional_wave_power': 'J (kW/m)', 'J_tm': 'J_tm (W/m)'
                       }
    summary_table['Parameter'] = summary_table['Parameter'].replace(parameter_names)

    fig, ax = plt.subplots(figsize = (16, 3))
    ax.axis('off')

    table = ax.table(cellText=summary_table.values, colLabels=summary_table.columns, cellLoc='center', loc='center'
                     )
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.8)
    ax.set_title(f"{names[j]} - Monthly Wave Statistics Summary")
    fig_path = os.path.join(graphics_folders[j], 'MonthlyStatisticsSummary.png')
    fig.savefig(fig_path, dpi = 300, bbox_inches='tight')
    plt.close(fig)
    # Needs to be Updated
    figure = plt.gcf()
    # exporting plot to png
    figure.set_size_inches(8, 10)
    fig.suptitle(f"{names[j]} - Monthly Averages")
    fig.tight_layout(rect=[0,0,1,0.96])
    fig_path = os.path.join(graphics_folders[j],'MonthlyAvgs.png')
    plt.savefig(fig_path, format = "png", dpi=100,bbox_inches='tight')
    plt.close()

--------------------------------------------
significant_wave_height max:0.7479, month: 4
significant_wave_height max mean:0.8181, month: 4
significant_wave_height min:0.3606, month: 8
significant_wave_height min mean0.4194, month: 8
--------------------------------------------
peak_period max:6.2925, month: 1
peak_period max mean:6.4505, month: 3
peak_period min:5.0, month: 7
peak_period min mean5.1183, month: 8
--------------------------------------------
energy_period max:5.1384, month: 3
energy_period max mean:4.989, month: 3
energy_period min:3.9833, month: 8
energy_period min mean4.1344, month: 8
--------------------------------------------
omni-directional_wave_power max:1491.575, month: 4
omni-directional_wave_power max mean:2386.9102, month: 3
omni-directional_wave_power min:282.9692, month: 8
omni-directional_wave_power min mean621.8273, month: 8
--------------------------------------------
significant_wave_height max:0.9975, month: 4
significant_wave_height max mean:1.0513, 

Plot Monthly Average Wave Flux in kW/m (Looped)

In [44]:
for i, D in enumerate(Ds):
    # Just plot Monthly Average Wave Flux in kW/m

    J = data['omni-directional_wave_power'] / 1000  # Convert from W/m to kW/m
    months = J.index.month
    data_group = J.groupby(months)
    stdd = data_group.std()

    fig, ax = plt.subplots(figsize=(12,5), sharex=True)

    # Plot mean
    ax.plot(data_group.mean(), marker='*', label='mean')

    # Shade between 10th and 90th percentile
    ax.fill_between(
        months.unique(),
        data_group.quantile(q=0.1),
        data_group.quantile(q=0.9),
        alpha=0.2,
        label='10-90th percentile'
    )
    ax.grid()
    # Axis labels and legend
    ax.set_xlabel('Month')
    ax.set_ylabel('$J$  [kW/m]')
    ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    ax.ticklabel_format(axis='y', style='plain')
    ax.set_ylim(0, 100)

    ax.set_title(f"{names[i]} - Monthly Average Wave Flux (kW/m)")
    # ax.set_title('Inshore - Monthly Average Wave Flux (kW/m)')
    plt.tight_layout()

    # Exporting plot to png
    figure = plt.gcf()
    figure.set_size_inches(8, 6)
    fig_path = os.path.join(graphics_folders[i], 'MonthlyAvg_J_kWm.png')
    plt.savefig(fig_path, format = "png", dpi=100, bbox_inches='tight')
    plt.close()


**Monthly Cumulative Distribution**

Monthly Cumulative Distribution (Looped)

In [45]:
for i, D in enumerate(Ds):
    ax = wave.graphics.monthly_cumulative_distribution(D['omni-directional_wave_power'])
    plt.xlim([500, 1E6])

    # Needs to be Updated
    figure = plt.gcf()
    plt.title(f"{names[i]} - Monthly Cumulative Distribution")
    # exporting plot to png
    figure.set_size_inches(8, 6)
    fig_path = os.path.join(graphics_folders[i],'MonthlyCumulativeDist.png') 
    plt.savefig(fig_path,format="png",dpi=100,bbox_inches='tight')
    plt.close()